In [0]:
import subprocess, sys, importlib
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "statsmodels"])
importlib.invalidate_caches()

import numpy as np, pandas as pd
import statsmodels.formula.api as smf

df = spark.table("retail_analysis.default.readmit_first_encounter").toPandas()
y = "target_readmit_early"
df["prior"] = np.where(df.number_inpatient >= 3, "3+", df.number_inpatient.astype(str))

d = pd.to_numeric(df.diag_1, errors="coerce")
df["dx"] = np.select(
    [df.diag_1.str.startswith("250", na=False), d.between(390,459)|(d==785), d.between(460,519)|(d==786),
     d.between(520,579)|(d==787), d.between(580,629)|(d==788), d.between(800,999),
     d.between(710,739), d.between(140,239)],
    ["Diabetes","Circulatory","Respiratory","Digestive","Genitourinary","Injury","Musculoskeletal","Neoplasms"], "Other")
df["age_grp"] = pd.cut(df.age.str.extract(r"\[(\d+)")[0].astype(int), [0,50,70,100], right=False, labels=["<50","50-69","70+"])

m = smf.logit(f"{y} ~ C(prior, Treatment('0')) + C(age_grp, Treatment('<50')) + C(dx, Treatment('Other'))"
              " + time_in_hospital + num_medications + number_diagnoses + number_emergency", df).fit(disp=0)
r = pd.DataFrame({"OR": np.exp(m.params), "low": np.exp(m.conf_int()[0]), "high": np.exp(m.conf_int()[1]), "p": m.pvalues}).round(3)
print(r)

                                                 OR    low   high      p
Intercept                                     0.041  0.036  0.047  0.000
C(prior, Treatment('0'))[T.1]                 1.555  1.431  1.690  0.000
C(prior, Treatment('0'))[T.2]                 2.364  2.066  2.704  0.000
C(prior, Treatment('0'))[T.3+]                3.696  3.169  4.312  0.000
C(age_grp, Treatment('<50'))[T.50-69]         1.098  1.007  1.197  0.035
C(age_grp, Treatment('<50'))[T.70+]           1.376  1.263  1.498  0.000
C(dx, Treatment('Other'))[T.Circulatory]      1.098  1.013  1.190  0.023
C(dx, Treatment('Other'))[T.Diabetes]         1.144  1.023  1.278  0.018
C(dx, Treatment('Other'))[T.Digestive]        0.920  0.824  1.027  0.136
C(dx, Treatment('Other'))[T.Genitourinary]    1.026  0.898  1.173  0.705
C(dx, Treatment('Other'))[T.Injury]           1.195  1.067  1.339  0.002
C(dx, Treatment('Other'))[T.Musculoskeletal]  1.031  0.903  1.177  0.652
C(dx, Treatment('Other'))[T.Neoplasms]        1.027

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

f = (f"{y} ~ C(prior, Treatment('0')) + C(age_grp, Treatment('<50')) + C(dx, Treatment('Other'))"
     " + time_in_hospital + num_medications + number_diagnoses + number_emergency")
tr, te = train_test_split(df, test_size=0.3, random_state=42, stratify=df[y])
te = te.assign(p=smf.logit(f, tr).fit(disp=0).predict(te))

print("AUC:", round(roc_auc_score(te[y], te.p), 3), "| PR-AUC:", round(average_precision_score(te[y], te.p), 3), "| baseline PR-AUC:", round(te[y].mean(), 3))
te["rank"] = te.p.rank(ascending=False, method="first")
for pct in [10, 20]:
    top = te[te["rank"] <= len(te) * pct / 100]
    print(f"Top {pct}% by risk: readmit rate {top[y].mean()*100:.1f}%, captures {top[y].sum()/te[y].sum()*100:.1f}% of all readmissions")

AUC: 0.607 | PR-AUC: 0.137 | baseline PR-AUC: 0.09
Top 10% by risk: readmit rate 16.0%, captures 17.8% of all readmissions
Top 20% by risk: readmit rate 13.7%, captures 30.6% of all readmissions
